# Transformación y homogeneización de datos

Este notebook se ejecuta de forma independiente del perfilado. Lee 2015–2019 desde
`data/raw` y produce dos CSV en `data/processed`, sin modificar los originales.

Decisiones acordadas:
- Nombres semánticos comunes en `snake_case` y columna `year`.
- Región sin espacios exteriores ni espacios repetidos; primera letra mayúscula
  y las demás minúsculas. Los nombres de países se conservan.
- Imputar el único nulo de corrupción de 2018 con la mediana de los valores
  observados **de ese mismo año**, antes de generar ambos datasets.
- Eliminar residuo de distopía, error estándar, intervalos de confianza y whiskers.
- Completar las regiones no disponibles con `Desconocido`, sin inferir una región geográfica.
- Mantener todos los países y los valores atípicos. No aplicar escalado numérico.

La imputación es una decisión de preparación adoptada para ambos usos. Su mediana
se calcula con el año completo, antes de una futura partición de modelado;
por tanto, la preparación no queda aislada de esa futura evaluación.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

YEARS = range(2015, 2020)
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / 'data' / 'raw' / f'{year}.csv').is_file() for year in YEARS)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontraron los cinco CSV en data/raw.')
raw_datasets = {year: pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / f'{year}.csv') for year in YEARS}
if any(df.empty for df in raw_datasets.values()):
    raise ValueError('Se encontró un dataset vacío.')
display(pd.DataFrame([{'year': year, 'rows': len(df), 'columns': len(df.columns)}
                      for year, df in raw_datasets.items()]))

   year  rows  columns
0  2015   158       12
1  2016   157       13
2  2017   155       12
3  2018   156        9
4  2019   156        9


## 1. Correspondencias y tipos

El diccionario resuelve diferencias de significado y escritura, además de usar
`snake_case`. Las columnas que luego se eliminarán se reconocen para validar el esquema de entrada.
La conversión numérica falla si encuentra texto inválido, en lugar de ocultarlo como nulo.

In [2]:
ALIASES = {'country': ['Country', 'Country or region'], 'region': ['Region'], 'happiness_rank': ['Happiness Rank', 'Happiness.Rank', 'Overall rank'], 'happiness_score': ['Happiness Score', 'Happiness.Score', 'Score'], 'gdp_per_capita': ['Economy (GDP per Capita)', 'Economy..GDP.per.Capita.', 'GDP per capita'], 'social_support': ['Family', 'Social support'], 'healthy_life_expectancy': ['Health (Life Expectancy)', 'Health..Life.Expectancy.', 'Healthy life expectancy'], 'freedom': ['Freedom', 'Freedom to make life choices'], 'corruption': ['Trust (Government Corruption)', 'Trust..Government.Corruption.', 'Perceptions of corruption'], 'generosity': ['Generosity'], 'dystopia_residual': ['Dystopia Residual', 'Dystopia.Residual'], 'standard_error': ['Standard Error'], 'lower_confidence_interval': ['Lower Confidence Interval'], 'upper_confidence_interval': ['Upper Confidence Interval'], 'whisker_low': ['Whisker.low'], 'whisker_high': ['Whisker.high']}

transformed = {}
schema_rows = []
for year, raw in raw_datasets.items():
    rename = {}
    for canonical, candidates in ALIASES.items():
        matches = [name for name in candidates if name in raw.columns]
        if len(matches) > 1:
            raise ValueError(f'{year}: varias columnas corresponden a {canonical}: {matches}')
        if matches:
            rename[matches[0]] = canonical
            schema_rows.append({'year': year, 'source': matches[0], 'canonical': canonical})
    unknown = set(raw.columns) - set(rename)
    if unknown:
        raise ValueError(f'{year}: columnas sin correspondencia: {unknown}')
    df = raw.rename(columns=rename).copy()
    df['country'] = df['country'].astype('string')
    if 'region' in df:
        df['region'] = (df['region'].astype('string').str.strip()
                        .str.replace(r'\s+', ' ', regex=True).str.capitalize())
    for column in df.columns.difference(['country', 'region']):
        df[column] = pd.to_numeric(df[column], errors='raise')
    df['happiness_rank'] = df['happiness_rank'].astype('Int64')
    df['year'] = year
    transformed[year] = df
display(pd.DataFrame(schema_rows).pivot(index='canonical', columns='year', values='source'))

year                                                2015  ...                          2019
canonical                                                 ...                              
corruption                 Trust (Government Corruption)  ...     Perceptions of corruption
country                                          Country  ...             Country or region
dystopia_residual                      Dystopia Residual  ...                           NaN
freedom                                          Freedom  ...  Freedom to make life choices
gdp_per_capita                  Economy (GDP per Capita)  ...                GDP per capita
generosity                                    Generosity  ...                    Generosity
happiness_rank                            Happiness Rank  ...                  Overall rank
happiness_score                          Happiness Score  ...                         Score
healthy_life_expectancy         Health (Life Expectancy)  ...       Healthy life

## 2. Imputación acordada para 2018

Se registra el país afectado y el valor aplicado. La mediana excluye el nulo.
Una comprobación exige que los datos coincidan con el hallazgo del perfilado;
si aparecen otros nulos, habrá que revisarlos antes de continuar.

In [3]:
missing_observed = [(year, column, int(count))
                    for year, df in transformed.items()
                    for column, count in df.isna().sum().items() if count]
if missing_observed != [(2018, 'corruption', 1)]:
    raise ValueError(f'Los nulos difieren del perfilado: {missing_observed}')
df_2018 = transformed[2018]
missing = df_2018['corruption'].isna()
affected_countries = df_2018.loc[missing, 'country'].tolist()
if affected_countries != ['United Arab Emirates']:
    raise ValueError(f'El país con nulo cambió: {affected_countries}')
corruption_median = float(df_2018['corruption'].median())
df_2018.loc[missing, 'corruption'] = corruption_median
imputation_log = {
    'year': 2018, 'column': 'corruption', 'method': 'median',
    'scope': 'observed values in 2018', 'value': corruption_median,
    'rows_imputed': int(missing.sum()), 'countries': affected_countries,
}
display(pd.DataFrame([imputation_log]))

   year      column  method  ...  value  rows_imputed               countries
0  2018  corruption  median  ...  0.082             1  [United Arab Emirates]

[1 rows x 7 columns]


## 3. Dataset analítico y dataset para modelos

`happiness_analysis.csv` conserva los indicadores comunes, el rango y la región.
Se eliminan `dystopia_residual`, `standard_error`, `lower_confidence_interval`,
`upper_confidence_interval`, `whisker_low` y `whisker_high`.
Las regiones no publicadas se etiquetan como `Desconocido`; no se infieren a partir de otros años.

`happiness_modeling.csv` contiene seis predictores, el objetivo y los identificadores
`country` y `year`. Estos identificadores permiten rastrear registros y definir la
partición; no se incluyen automáticamente como predictores. Se excluye el rango
porque deriva del puntaje. La región tampoco se usa como predictor.


In [4]:
FEATURES = ['gdp_per_capita', 'social_support', 'healthy_life_expectancy',
            'freedom', 'generosity', 'corruption']
TARGET = 'happiness_score'
IDENTIFIERS = ['country', 'year']
DROP_COLUMNS = ['dystopia_residual', 'standard_error', 'lower_confidence_interval',
                'upper_confidence_interval', 'whisker_low', 'whisker_high']
analysis = pd.concat(transformed.values(), ignore_index=True, sort=False)
analysis = analysis.reindex(columns=['country', 'year'] + [c for c in ALIASES if c != 'country'])
analysis = analysis.drop(columns=DROP_COLUMNS)
region_missing_count = int(analysis['region'].isna().sum())
analysis['region'] = analysis['region'].astype('string').fillna('Desconocido')
analysis['year'] = analysis['year'].astype('Int64')
modeling = analysis[IDENTIFIERS + FEATURES + [TARGET]].copy()
display(analysis.head())
display(modeling.head())

       country  year          region  ...  freedom  corruption  generosity
0  Switzerland  2015  Western europe  ...  0.66557     0.41978     0.29678
1      Iceland  2015  Western europe  ...  0.62877     0.14145     0.43630
2      Denmark  2015  Western europe  ...  0.64938     0.48357     0.34139
3       Norway  2015  Western europe  ...  0.66973     0.36503     0.34699
4       Canada  2015   North america  ...  0.63297     0.32957     0.45811

[5 rows x 11 columns]
       country  year  gdp_per_capita  ...  generosity  corruption  happiness_score
0  Switzerland  2015         1.39651  ...     0.29678     0.41978            7.587
1      Iceland  2015         1.30232  ...     0.43630     0.14145            7.561
2      Denmark  2015         1.32548  ...     0.34139     0.48357            7.527
3       Norway  2015         1.45900  ...     0.34699     0.36503            7.522
4       Canada  2015         1.32629  ...     0.45811     0.32957            7.427

[5 rows x 9 columns]


## 4. Validación antes de guardar

Verificamos conservación de filas, clave país–año, columnas, tipos y ausencia de nulos
en ambos datasets. Las regiones no disponibles deben contener `Desconocido`.


In [5]:
expected_rows = sum(len(df) for df in raw_datasets.values())
assert len(analysis) == len(modeling) == expected_rows
assert not analysis.duplicated(['country', 'year']).any(), 'Clave país–año duplicada'
assert not analysis['country'].isna().any()
assert analysis['country'].str.strip().ne('').all()
assert not modeling.isna().any(axis=None), 'El dataset de modelado contiene nulos'
assert not analysis.isna().any(axis=None), 'El dataset analítico contiene nulos'
assert not set(DROP_COLUMNS).intersection(analysis.columns)
assert analysis['region'].eq('Desconocido').sum() == region_missing_count
assert analysis.columns.is_unique
assert analysis.columns.str.fullmatch(r'[a-z][a-z0-9_]*').all()
assert all(pd.api.types.is_numeric_dtype(modeling[c]) for c in FEATURES + [TARGET, 'year'])
for year, raw in raw_datasets.items():
    subset = analysis.loc[analysis['year'].eq(year)]
    assert len(subset) == len(raw)
    available = set(transformed[year].columns)
    for column in analysis.columns:
        if column in available:
            assert subset[column].notna().all(), (year, column)
        else:
            assert column == 'region', (year, column)
            assert subset[column].eq('Desconocido').all(), (year, column)
    if 'region' in available:
        region = subset['region']
        assert region.eq(region.str.strip().str.replace(r'\s+', ' ', regex=True).str.capitalize()).all()
display(analysis.isna().sum().rename('missing_values').to_frame())
print(f'Validación correcta: {expected_rows} filas; análisis {analysis.shape}, modelado {modeling.shape}.')

                         missing_values
country                               0
year                                  0
region                                0
happiness_rank                        0
happiness_score                       0
gdp_per_capita                        0
social_support                        0
healthy_life_expectancy               0
freedom                               0
corruption                            0
generosity                            0
Validación correcta: 782 filas; análisis (782, 11), modelado (782, 9).


In [6]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {'happiness_analysis.csv': analysis, 'happiness_modeling.csv': modeling}
for filename, df in outputs.items():
    destination = OUTPUT_DIR / filename
    df.to_csv(destination, index=False, encoding='utf-8')
    reloaded = pd.read_csv(destination)
    pd.testing.assert_frame_equal(df.reset_index(drop=True).convert_dtypes(), reloaded.convert_dtypes(), check_dtype=False,
                                  check_exact=False, rtol=1e-12, atol=1e-12)
    print(f'Guardado y verificado: {destination.name} ({len(reloaded)} filas)')
metadata = {'years': list(YEARS), 'rows': len(analysis), 'features': FEATURES,
            'target': TARGET, 'identifiers': IDENTIFIERS, 'imputation': imputation_log,
            'dropped_columns': DROP_COLUMNS,
            'region_fill': {'value': 'Desconocido', 'rows_filled': region_missing_count},
            'missing_values': analysis.isna().sum().astype(int).to_dict()}
(OUTPUT_DIR / 'transformation_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

Guardado y verificado: happiness_analysis.csv (782 filas)
Guardado y verificado: happiness_modeling.csv (782 filas)


## Resultado

Ambos datasets conservan las filas y la imputación de corrupción de 2018, sin nulos.
El analítico reemplaza las regiones ausentes por `Desconocido` y excluye las seis
variables acordadas. `transformation_metadata.json` documenta las decisiones y los conteos.
Los CSV no conservan los tipos anulables de pandas; al leerlos de nuevo conviene
declarar explícitamente los tipos cuando se necesiten.

Se tuvo en cuenta los rangos de las variables: https://www.kaggle.com/datasets/unsdsn/world-happiness/data

